# Sentiment and Product Entities

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/04_text_classification_exercise.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

A sentiment pipeline tells you **whether** a review is positive or negative. A NER pipeline tells you **which product** that sentiment is about. This lab chains both Hugging Face encoder pipelines.


**Goal:** For each review, classify sentiment and extract the product name — then report which products are praised and which are criticized.

**Topics:** `pipeline("sentiment-analysis")`, `pipeline("ner")`, aggregation strategies, pairing document labels with token entities.


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/pipelines/02-pipelines"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Imports


In [ ]:
%pip install -qqq transformers


In [ ]:
from transformers import pipeline


# NLP Scenario
You are an analyst for a marketing company that just launched a **product suite** of mobile devices. The client does not want a single sentiment score for the whole line — they want to know **which product** each review is about, and whether that mention is positive or negative.

##### Product Reviews
1. "I absolutely love the TechWave X1! It has made my daily tasks in Spain so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the Pulse Buds. They lack bass and disconnect constantly here in Mexico."
3. "The Nova Tablet is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The Pulse Watch is terrible. The battery dies by noon and the band irritates my skin. I regret purchasing it."
6. "The Nova Tablet is disappointing. It doesn't live up to the hype and is missing several key functionalities."


In [ ]:
reviews = [
    "I absolutely love the TechWave X1! It has made my daily tasks in Spain so much easier and more efficient. Highly recommend it!",
    "I'm not very impressed with the Pulse Buds. They lack bass and disconnect constantly here in Mexico.",
    "The Nova Tablet is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
    "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
    "The Pulse Watch is terrible. The battery dies by noon and the band irritates my skin. I regret purchasing it.",
    "The Nova Tablet is disappointing. It doesn't live up to the hype and is missing several key functionalities.",
]


# Sentiment analysis

The default analyzer for sentiment can be called with:
```python
pipeline("sentiment-analysis")
```
This pipeline takes a sentence, paragraph, or document and returns a sentiment score and a label:
```python
[{'score': 0.9991, 'label': 'POSITIVE'}]
```

The score is the model's confidence. Closer to 1.0 means a more confident label.

While it downloads, skim the [default DistilBERT SST-2 model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).


In [ ]:
sentiment_analyzer = pipeline("sentiment-analysis")


In [ ]:
sentiment_analyzer(reviews[0])


# Try it!
1. Run sentiment analysis on **all six** reviews.
2. Compare the labels to your own reading. How did the model treat the average review (item 4)?


In [ ]:
sentiment_analyzer(reviews)

# Named Entity Recognition (NER)

Sentiment is a **document** label. Product names live **inside** the text. Token classification (NER) labels each token.

The default analyzer for NER can be called with:
```python
pipeline("ner")
```
Each hit includes:
- `entity` — classification (see types below)
- `score` — confidence (0–1)
- `index` — token position
- `word` — surface form (may be a subword such as `##wave`)
- `start` / `end` — character offsets

#### Entity types
- `O` — not an entity
- `B-PER` / `I-PER` — beginning of / inside a person
- `B-ORG` / `I-ORG` — organization
- `B-LOC` / `I-LOC` — location
- `B-MISC` / `I-MISC` — miscellaneous (often product names)

Source: [Hugging Face NLP course](https://huggingface.co/learn/nlp-course/en/chapter7/2?fw=pt)


In [ ]:
ner_pipeline = pipeline("ner")


While you're waiting for the model to download, open its model card on Hugging Face.


In [ ]:
results = ner_pipeline(reviews[0])
for entity in results:
    print(f"Entity: {entity['entity']}, Value: {entity['word']}")


Product names such as TechWave are often split into WordPiece subwords (`tech`, `##wave`). Locations such as Spain usually come through as `LOC`.

Set `aggregation_strategy="simple"` to merge subwords into whole entities and drop the `B-` / `I-` prefixes:

```python
ner_simple = pipeline("ner", aggregation_strategy="simple")
```


In [ ]:
ner_simple = pipeline("ner", aggregation_strategy="simple")

In [ ]:
def extract_products(review: str) -> list[str]:
  outputs = ner_simple(review)
  results = []
  for out in outputs:
    if out['entity_group'] == 'MISC' or out['entity_group'] == 'ORG':
      results.append(out['word'])
  return results

extract_products(reviews[0])

# Try it!
The client wants a **product-level** report, not a pile of review labels.

1. Run `sentiment_analyzer` and `ner_simple` on **all six** reviews.
2. For each review, pair the sentiment `label` with the product entity (usually `MISC` or `ORG` — ignore `LOC`).
3. Split the results:
   - **Positive** — which product was praised?
   - **Negative** — which product was criticized?
4. How did the model treat the average TechWave X1 review? Did NER catch every product name (`TechWave X1`, `Pulse Buds`, `Nova Tablet`, `Pulse Watch`)?


In [ ]:
sentiment_outputs = sentiment_analyzer(reviews)
sentiment_outputs

In [ ]:
products_reviewed = []
for r in reviews:
  result = extract_products(r)
  products_reviewed.append(result)

products_reviewed

In [ ]:
import pandas as pd

# Prepare data by pairing sentiment and products for each review
data = []
for i in range(len(reviews)):
    data.append({
        "Review": reviews[i],
        "Sentiment": sentiment_outputs[i]['label'],
        "Products": ", ".join(products_reviewed[i])
    })

# Create and display the DataFrame
df = pd.DataFrame(data)
df

In [ ]:
df.groupby('Products').value_counts()

## Conclusion
- Hugging Face pipelines let you prototype encoder tasks with a one-line `pipeline(...)` call
- Sentiment is a document label; NER is a token label — you need both to say *which product* a review is about
- `aggregation_strategy="simple"` merges WordPiece subwords into whole entities
- Off-the-shelf NER is trained on news entities (people, places, organizations). Product names are often `MISC`, split, or missed — check the output before you trust a product-level report